# ALS (Alternating Least Squares) разложение

Необходимо по истории взаимодействий пользователя предсказать товары, которые он закажет в ближайшие 2 недели.

Для подбора гиперпараметров из обучающей выборки последние 2 недели выделены под валидацию (с 18 июня 2025 по 1 июля 2025)

Было сформировано 2 варианта матрицы `users × items`:
- на всех доступных данных (с 1 января 2025 по 17 июня 2025)
- на данных за последние два месяца (с 18 апреля 2025 по 17 июня 2025)

Действия пользователя разделены на группы:
- **Order** - (явная покупка, максимальный вес в матрице)
- **Strong** - явные действия намерения, но не гарантия покупки (добавление в корзину / избранное)
- **Weak** - исследовательское поведение (просмотр товара, опичание, отзывов) - накапливается до определенного порога

Полученные метрики:
- NDCG@100 on val: 0.00997
- NDCG@100 on test: 0.01183

Примечания:
- сокращение окна обучения с 6,5 до 2 месяцев существенно улучшило метрику
- увеличение factors с 64 до 128 существенно улучшило метрику (на 256 ядро падает)
- усиление регуляризации незначительно ухудшило метрику
- применение bm25_weight незначительно ухудшило метрику

Зона для улучшения:
- изменение весов событий (очень долго, нужен пересбор матрицы `users × items`)
- ослабление week событий со временем (очень долго, нужен пересбор матрицы `users × items`)

In [1]:
import os
import sys
from collections import defaultdict
from datetime import date
from functools import reduce
from pathlib import Path
from tqdm import trange

import dask.dataframe as dd
import implicit
import numpy as np
import pandas as pd
from implicit.als import AlternatingLeastSquares
from scipy.sparse import csr_matrix, save_npz, load_npz

sys.path.append('../src')

from data_loader import (
    download_dataset,
    load_orders,
    load_interactions,
    load_items
)

os.environ["OPENBLAS_NUM_THREADS"] = "1"

In [3]:
download_dataset()

Датасет загружен ранее


In [4]:
TRAIN_START = date(2025, 4, 18)
TRAIN_END  = date(2025, 6, 17)

VAL_START  = date(2025, 6, 18)
VAL_END    = date(2025, 7, 1)

WEAK_EVENTS   = {"page_view", "view_description", "review_view"}
STRONG_EVENTS = {"to_cart", "favorite"}

# Веса событий (WEAK_EVENTS накапливаются до WEAK_MAX_CAP)
EVENT_WEIGHTS = {
    "page_view": 1,
    "view_description": 2,
    "review_view": 3,
    "favorite": 20,
    "to_cart": 40,
    "order": 100,
}

# Ограничение кумулятивного веса для WEAK_EVENTS
WEAK_MAX_CAP = 15

# Количество шардов для сохранения матрицы users × items
N_SHARDS = 128    # если упадёт — 256  # на 64 упало на 6 месяцах

# Директория для сохранения шардов
OUT_DIR = f"../data/processed/csr_build_{TRAIN_START}_{TRAIN_END}"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

## Формироване матрицы **users × items**

In [4]:
def filter_tracker_ddf(
    tracker_ddf: dd.DataFrame,
    start_date: date,
    end_date: date
) -> dd.DataFrame:
    """
    Фильтрация действий пользователей по времени
    """

    tracker_ddf = tracker_ddf.assign(
        date=dd.to_datetime(tracker_ddf["date"], format="%Y-%m-%d")
    )

    return tracker_ddf[
        (tracker_ddf["date"] >= pd.to_datetime(start_date)) &
        (tracker_ddf["date"] <= pd.to_datetime(end_date)) &
        (tracker_ddf["action_type"].isin(WEAK_EVENTS | STRONG_EVENTS))
    ][["user_id", "item_id", "action_type"]]


In [5]:
def filter_orders_ddf(
    orders_ddf: dd.DataFrame,
    start_date: date,
    end_date: date
) -> dd.DataFrame:
    """
    Фильтрация заказов пользователей пользователя по времени
    """

    orders_ddf = orders_ddf.assign(
        created_date=dd.to_datetime(orders_ddf["created_date"], format="%Y-%m-%d")
    )

    return orders_ddf[
        (orders_ddf["created_date"] >= pd.to_datetime(start_date)) &
        (orders_ddf["created_date"] <= pd.to_datetime(end_date))
    ][["user_id", "item_id"]]


In [6]:
def build_id_mappings(
    tracker_ddf: dd.DataFrame,
    orders_ddf: dd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Формирование маппингов реальных id пользователя/товара на idx для ALS
    """

    users = dd.concat([
        tracker_ddf[["user_id"]],
        orders_ddf[["user_id"]],
    ]).drop_duplicates().compute()

    items = dd.concat([
        tracker_ddf[["item_id"]],
        orders_ddf[["item_id"]],
    ]).drop_duplicates().compute()

    users = users["user_id"].astype("int64").sort_values().reset_index(drop=True)
    items = items["item_id"].astype("int64").sort_values().reset_index(drop=True)

    user_mapping = pd.DataFrame({
        "user_id": users,
        "user_idx": np.arange(len(users), dtype=np.int32),
    })

    item_mapping = pd.DataFrame({
        "item_id": items,
        "item_idx": np.arange(len(items), dtype=np.int32),
    })

    return user_mapping, item_mapping


In [7]:
def build_interactions_ddf(
    tracker_pdf: pd.DataFrame,
    orders_pdf: pd.DataFrame,
) -> pd.DataFrame:
    """
    Формирование датафрейма user-item-weight
    """

    tracker_pdf["weight"] = (
        tracker_pdf["action_type"]
        .map(EVENT_WEIGHTS)
        .astype("int8")
    )

    weak = (
        tracker_pdf[tracker_pdf["action_type"].isin(WEAK_EVENTS)]
        .groupby(["user_id", "item_id"], as_index=False)
        .agg(weight=("weight", "sum"))
    )
    weak["weight"] = weak["weight"].clip(upper=WEAK_MAX_CAP)

    strong = (
        tracker_pdf[tracker_pdf["action_type"].isin(STRONG_EVENTS)]
        .groupby(["user_id", "item_id"], as_index=False)
        .agg(weight=("weight", "max"))
    )

    orders = (
        orders_pdf
        .drop_duplicates()
        .assign(weight=EVENT_WEIGHTS["order"])
        .astype({"weight": "int8"})
    )

    all_events = pd.concat([weak, strong, orders], ignore_index=True)

    interactions = (
        all_events
        .groupby(["user_id", "item_id"], as_index=False)
        .agg(weight=("weight", "max"))
    )

    return interactions.astype({
        "user_id": "int32",
        "item_id": "int32",
        "weight": "int8",
    })


In [8]:
def process_shard(
    shard_id: int,
    n_shards: int,
    tracker_ddf: dd.DataFrame,
    orders_ddf: dd.DataFrame,
    u2i: dict,
    i2i: dict,
    n_users: int,
    n_items: int,
    out_dir: str,
):
    """
    Формирование матрицы взаимодействий user-item для шарда
    """

    tracker_pdf = tracker_ddf[
        (tracker_ddf["user_id"] % n_shards) == shard_id
    ].compute()

    orders_pdf = orders_ddf[
        (orders_ddf["user_id"] % n_shards) == shard_id
    ].compute()

    if tracker_pdf.empty and orders_pdf.empty:
        return

    interactions = build_interactions_ddf(tracker_pdf, orders_pdf)

    interactions["u"] = interactions["user_id"].map(u2i)
    interactions["i"] = interactions["item_id"].map(i2i)

    interactions.dropna(subset=["u", "i"], inplace=True)

    mat = csr_matrix(
        (
            interactions["weight"].astype(np.float32).values,
            (interactions["u"].values.astype("int32"),
             interactions["i"].values.astype("int32")),
        ),
        shape=(n_users, n_items),
    )

    save_npz(f"{out_dir}/shard_{shard_id}.npz", mat)


In [9]:
def build_interaction_csr(
    tracker_ddf: dd.DataFrame,
    orders_ddf: dd.DataFrame,
    start_date,
    end_date,
    n_shards: int,
    out_dir: str,
):
    """
    Формирование матриц взаимодействий user-item по шардам
    и сохранение маппингов
    """
    tracker_ddf = filter_tracker_ddf(
        tracker_ddf, start_date, end_date
    )
    orders_ddf = filter_orders_ddf(
        orders_ddf, start_date, end_date
    )

    user_mapping, item_mapping = build_id_mappings(
        tracker_ddf, orders_ddf
    )

    u2i = dict(zip(user_mapping["user_id"], user_mapping["user_idx"]))
    i2i = dict(zip(item_mapping["item_id"], item_mapping["item_idx"]))

    n_users = len(user_mapping)
    n_items = len(item_mapping)

    user_mapping.to_parquet(f"{out_dir}/user_mapping.parquet", index=False)
    item_mapping.to_parquet(f"{out_dir}/item_mapping.parquet", index=False)

    for shard_id in range(n_shards):
        print(f"Shard {shard_id + 1}/{n_shards}")
        process_shard(
            shard_id,
            n_shards,
            tracker_ddf,
            orders_ddf,
            u2i,
            i2i,
            n_users,
            n_items,
            out_dir,
        )

In [10]:
def build_final_csr(out_dir: str, n_shards: int):
    """
    Сборка csr матрицы user-item из сохраненных ранее шардов
    """
    mats = []
    for k in range(n_shards):
        try:
            mats.append(load_npz(f"{out_dir}/shard_{k}.npz"))
        except FileNotFoundError:
            pass

    final = reduce(lambda a, b: a + b, mats)
    save_npz(f"{out_dir}/interaction_matrix.npz", final)
    return final

In [22]:
tracker_ddf = load_interactions(train=True)
orders_ddf = load_orders(train=True)

build_interaction_csr(
    tracker_ddf=tracker_ddf,
    orders_ddf=orders_ddf,
    start_date=TRAIN_START,
    end_date=TRAIN_END,
    n_shards=N_SHARDS,
    out_dir=OUT_DIR,
)

Shard 1/128


In [ ]:
user_items_csr = build_final_csr(
    out_dir=OUT_DIR,
    n_shards=N_SHARDS,
)

user_items_csr

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 279714816 stored elements and shape (954115, 4321191)>

## Обучение ALS

In [44]:
user_items_csr = load_npz(f"{OUT_DIR}/interaction_matrix.npz")
user_items_csr    # users × items

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 279714816 stored elements and shape (954115, 4321191)>

In [45]:
model = AlternatingLeastSquares(
    factors=128,
    regularization=0.01,
    iterations=20,
    use_gpu=False,
)

model.fit(user_items_csr)

  0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
model.save("../models/als_model_for_val.npz")

In [47]:
assert user_items_csr.shape[0] == model.user_factors.shape[0]
assert user_items_csr.shape[1] == model.item_factors.shape[0]

In [48]:
orders_ddf = load_orders(train=True)
orders_ddf.head()

,item_id,user_id,created_timestamp,last_status,last_status_timestamp,created_date
0,15535,3423970,2025-01-11 19:20:02.550,delivered_orders,2025-01-16 10:53:03,2025-01-11
1,15535,1207601,2025-03-25 13:11:48.240,canceled_orders,2025-03-29 08:32:43,2025-03-25
2,119241,2340021,2025-05-03 20:46:18.146,canceled_orders,2025-05-12 17:10:27,2025-05-03
3,119241,2356390,2025-05-01 17:53:00.716,delivered_orders,2025-05-14 14:36:20,2025-05-01
4,119241,1490080,2025-04-22 22:59:11.160,canceled_orders,2025-04-26 06:27:28,2025-04-22


In [49]:
user_map = pd.read_parquet(f"{OUT_DIR}/user_mapping.parquet")

user_id = 3423970
u_idx = user_map.loc[user_map.user_id == user_id, "user_idx"].iloc[0]
u_idx

np.int32(645473)

In [50]:
items, scores = model.recommend(
    userid=u_idx,
    user_items=user_items_csr[u_idx],
    N=100,
    filter_already_liked_items=True,
)
items

array([3688153,  522736, 3134226, 2319828, 1387814, 3949570,  446051,
       3070217, 4291117, 2538414, 4201948, 2634326, 1977675, 1992149,
        366781,    2989, 3563007,  113667, 1906252, 3985937, 1056205,
        424147,  951775, 3642618, 2420963,  595087, 1250076, 4033372,
        963968, 4095644,  742779,  908860, 2156409, 1896199, 3769638,
        360109,  982093, 2123974, 3604364, 1037894, 1706789,  989435,
        146405, 2259715, 4101708, 1902334, 4295605, 4066732, 3130715,
        927642, 3755379, 3377572,  939775,  422618,  312170, 3058331,
        870138, 2202476, 2431664,  561125, 4091224, 1942157, 2494906,
       3567225, 1890857, 3363902,  943028,  178136,  739797,  996649,
        373382,  334454, 3905257,  446901, 3117120, 2384077, 3148608,
         21080, 3024502, 4294317, 1755714, 1546260, 1262661,   79792,
        177104, 4314970,  412288,  163346, 1861941, 2130586, 3077730,
        732147, 3613238, 2017763, 1066411,  386449, 3440266, 2059149,
       1301234, 1103

In [51]:
item_map = pd.read_parquet(f"{OUT_DIR}/item_mapping.parquet")
items_ddf = load_items()

item_id = item_map[item_map['item_idx'].isin(items)]['item_id']

recs = items_ddf[items_ddf['item_id'].isin(item_id)].compute()
recs

,item_id,itemname,attributes,fclip_embed,catalogid,variant_id,model_id
40132,77109546,Кроссовки Xcomfy,"[{'attribute_name': 'Name', 'attribute_value':...","[-0.13016401, 1.6425232, -0.6323788, -0.072021...",36484,6781546,31328142
61894,58318760,Сарафан Harper's Bazaar (журнал),"[{'attribute_name': 'Type', 'attribute_value':...","[-0.05972548, 1.671419, 0.86633307, -0.1296886...",36741,24984187,37023602
37285,26253958,Брюки Натали Суслова Большие размеры,"[{'attribute_name': 'Type', 'attribute_value':...","[0.17031057, 0.39096522, 0.14132522, 0.0547992...",7512,282309928,25118189
39944,24511368,Брюки Натали Суслова Большие размеры,"[{'attribute_name': 'Type', 'attribute_value':...","[0.30463493, 0.3723288, 0.16868816, 0.10892106...",7512,229707216,25118189
42654,44053675,Платье KVARTALL Лето,"[{'attribute_name': 'Type', 'attribute_value':...","[0.065433115, 1.0824628, -0.03450963, 0.166328...",7502,156989405,177490
...,...,...,...,...,...,...,...
53264,71331024,Платье MISS-TEX,"[{'attribute_name': 'Type', 'attribute_value':...","[0.66660017, 0.88036287, -0.009014089, -0.2783...",7502,267229597,12487751
30193,240115287,Футболка Newlife базовая большого размера хлопок,"[{'attribute_name': 'Type', 'attribute_value':...","[0.40243912, 1.2606403, 0.21957318, 0.37103367...",7508,42025723,17961543
27965,265274431,Футболка Утк-style,"[{'attribute_name': 'Type', 'attribute_value':...","[-0.81452894, 2.0253701, -0.2354626, 0.0665551...",7508,292117667,36487442
28251,237179,Футболка Апрель,"[{'attribute_name': 'Type', 'attribute_value':...","[-0.18020874, 1.0694798, -0.17670688, 0.144358...",7508,138219923,14882893


## Оценка NDCG@100

In [68]:
orders_ddf = load_orders(train=True)
user_items_csr = load_npz(f"{OUT_DIR}/interaction_matrix.npz")
model = implicit.cpu.als.AlternatingLeastSquares.load("../models/als_model_for_val.npz")

### Подготовка ground truth для валидации (реальные покупки пользователей)

In [70]:
def make_df_for_eval(orders_ddf_eval, out_dir, start_date, end_date):
    """
    Получение выборки для оценки ALS:
    - клиент делал заказы в период оценки
    - клиент присутствует в обучающей выборке
    """
    user_map = pd.read_parquet(f"{out_dir}/user_mapping.parquet")
    item_map = pd.read_parquet(f"{out_dir}/item_mapping.parquet")

    user2idx = dict(zip(user_map.user_id, user_map.user_idx))
    item2idx = dict(zip(item_map.item_id, item_map.item_idx))

    orders_ddf_eval = orders_ddf_eval.assign(
        created_date=dd.to_datetime(orders_ddf_eval['created_date'])
    )
    
    eval_df = orders_ddf_eval[
        (orders_ddf_eval['created_date'] >= pd.to_datetime(start_date)) &
        (orders_ddf_eval['created_date'] <= pd.to_datetime(end_date))][['user_id', 'item_id']].compute()

    eval_df = eval_df[
        eval_df.user_id.isin(user2idx) &
        eval_df.item_id.isin(item2idx)
    ].copy()

    eval_df["u"] = eval_df.user_id.map(user2idx)
    eval_df["i"] = eval_df.item_id.map(item2idx)

    gt = defaultdict(set)
    for u, i in zip(eval_df.u, eval_df.i):
        gt[u].add(i)

    users_for_eval = eval_df['u'].unique().tolist()

    return eval_df, gt, users_for_eval

In [71]:
eval_df, gt, users_for_eval = make_df_for_eval(
    orders_ddf, OUT_DIR, VAL_START, VAL_END)

display(eval_df.head())
print("Users in val GT:", len(gt))
print("Avg GT size:", np.mean([len(v) for v in gt.values()]))

,user_id,item_id,u,i
9,2902640,191536,547084,2402
11,1111231,191536,209573,2402
15,898360,219368,169461,2743
25,4959670,320151,934894,4056
26,1623550,320379,306104,4060


Users in val GT: 355661
Avg GT size: 4.321387501019229


### Ранее купленные товары пользавателями

In [53]:
def build_purchased_by_user_idx(
    orders_ddf: dd.DataFrame,
    user_map: pd.DataFrame,
    item_map: pd.DataFrame,
    start_date,
    end_date,
):
    """
    OUTPUT:
        dict[user_idx] -> np.array(item_idx, dtype=int32)
    """

    orders_ddf = orders_ddf.assign(
        created_date=dd.to_datetime(
            orders_ddf["created_date"], format="%Y-%m-%d"
        )
    )

    orders_ddf = orders_ddf[
        (orders_ddf["created_date"] >= pd.to_datetime(start_date)) &
        (orders_ddf["created_date"] <= pd.to_datetime(end_date))
    ][["user_id", "item_id"]]

    orders_ddf = orders_ddf.astype({
        "user_id": "int64",
        "item_id": "int64",
    })

    user_map = user_map.astype({"user_id": "int64"})
    item_map = item_map.astype({"item_id": "int64"})

    tmp = (
        orders_ddf
        .merge(user_map[["user_id", "user_idx"]], on="user_id", how="inner")
        .merge(item_map[["item_id", "item_idx"]], on="item_id", how="inner")
        [["user_idx", "item_idx"]]
        .drop_duplicates()
        .compute()
    )

    purchased = defaultdict(list)

    for u, i in zip(tmp.user_idx.values, tmp.item_idx.values):
        purchased[u].append(i)

    return {
        u: np.array(items, dtype=np.int32)
        for u, items in purchased.items()
    }

In [72]:
user_map = pd.read_parquet(f"{OUT_DIR}/user_mapping.parquet")
item_map = pd.read_parquet(f"{OUT_DIR}/item_mapping.parquet")

purchased_by_user = build_purchased_by_user_idx(
    orders_ddf, user_map, item_map, TRAIN_START, TRAIN_END)

purchased_by_user

{np.int32(654152): array([   1478, 3118308, 4072542, 1529720, 2951966, 2006320, 4185670,
         986078, 3484299, 1052647, 2011046, 2859301, 2244314,  703199,
        2158065, 1526358, 1768107,  679447, 1201545, 3745848, 3634256,
        1360487, 1071508,   32386, 3214367,  986303, 2720038,  799309,
         208517,  387591, 3768370, 2027354, 2499134, 3616952, 2746806,
         576624, 2291235, 1140539, 2013145,   42362], dtype=int32),
 np.int32(793162): array([   1930, 4308433,  531648, 2471106,  468299, 2173777, 1148483,
        1141491, 1609023,  660141,  568858, 2620732, 3580413, 4286765,
        3727783,  691084, 3321489, 2939692, 1530941, 3474268, 3898118,
          93785, 1074424, 2528661, 4139086, 3212812, 3886212,  700846,
        1449692, 1141532, 1445800, 3798268, 1619440, 1620529, 3487637,
        4151125, 4149560, 3934194,  392501, 1113542, 3588169, 1694647,
        4206035, 2681457, 1037273, 3075670, 3794290, 1710990, 1136707,
        3504197, 2503802,  128578, 2265393, 

In [ ]:
normalized_purchased_by_user_idx = {
    int(user_idx): items.astype(np.int32, copy=False)
    for user_idx, items in purchased_by_user.items()
}

np.save(
    f"{OUT_DIR}/als_purchased_by_user_idx.npy"
    normalized_purchased_by_user_idx,
    allow_pickle=True
)

### Оценка работы модели

In [57]:
def batch_recommend_no_filter(
    model,
    matrix,
    user_idxs,
    k,
):
    """
    Возвращает raw top-k рекомендаций без фильтрации
    """
    user_items = matrix[user_idxs]

    item_idxs, _ = model.recommend(
        userid=user_idxs,
        user_items=user_items,
        N=k,
        filter_already_liked_items=False,
    )

    return item_idxs   # batch x k


In [58]:
def postprocess_recommendations(
    batch_users,
    batch_items,
    purchased_by_user,
    k,
):
    """
    Исключение из рекомендаций ранее купленных товаров
    """
    results = {}

    for u, recs in zip(batch_users, batch_items):
        purchased = purchased_by_user.get(u)
        if purchased is not None:
            purchased = set(purchased)
            recs = [i for i in recs if i not in purchased]

        results[u] = np.array(recs[:k], dtype=np.int32)

    return results


In [59]:
def ndcg_at_k(recommended, relevant, k):
    if not relevant:
        return 0.0

    dcg = 0.0
    for rank, item in enumerate(recommended[:k], start=1):
        if item in relevant:
            dcg += 1.0 / np.log2(rank + 1)

    idcg = sum(
        1.0 / np.log2(i + 1)
        for i in range(1, min(len(relevant), k) + 1)
    )

    return dcg / idcg if idcg > 0 else 0.0


In [60]:
def evaluate_ndcg(
    model,
    matrix,
    users_for_eval,
    gt,
    purchased_by_user,
    k=100,
    batch_size=512,
    oversample=1000
):
    scores = []

    for i in trange(0, len(users_for_eval), batch_size):
        batch_users = np.array(
            users_for_eval[i:i + batch_size],
            dtype=np.int32,
        )

        # Рекомендации с запасом (для последующего исключения купленных товаров)
        batch_items = batch_recommend_no_filter(
            model,
            matrix,
            batch_users,
            k=oversample,
        )

        # Исключение купленных товаров
        batch_recs = postprocess_recommendations(
            batch_users,
            batch_items,
            purchased_by_user,
            k,
        )

        # Расчет NDCG@100
        for u in batch_users:
            scores.append(
                ndcg_at_k(
                    batch_recs[u],
                    gt[u],
                    k,
                )
            )

    return float(np.mean(scores))

In [73]:
NDCG_100 = evaluate_ndcg(
    model=model,
    matrix=user_items_csr,
    users_for_eval=users_for_eval,
    gt=gt,
    purchased_by_user=purchased_by_user,
    k=100,
    batch_size=512,
    oversample=1000,
)

print(f"NDCG@100: {NDCG_100:.5f}")


100%|██████████| 695/695 [26:05<00:00,  2.25s/it]

NDCG@100: 0.00997


## Обучение модели на новом периоде и оценка на тесте

### Формироване матрицы **users × items**

In [27]:
NEW_TRAIN_START = date(2025, 5, 2)
NEW_TRAIN_END  = date(2025, 7, 1)

NEW_VAL_START = date(2025, 7, 2)
NEW_VAL_END  = date(2025, 7, 15)

# Директория для сохранения шардов
OUT_DIR = f"../data/processed/csr_build_{NEW_TRAIN_START}_{NEW_TRAIN_END}"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

In [18]:
tracker_ddf = load_interactions(train=True)
orders_ddf = load_orders(train=True)

build_interaction_csr(
    tracker_ddf=tracker_ddf,
    orders_ddf=orders_ddf,
    start_date=NEW_TRAIN_START,
    end_date=NEW_TRAIN_END,
    n_shards=N_SHARDS,
    out_dir=OUT_DIR,
)

Shard 86/128
Shard 87/128
Shard 88/128
Shard 89/128
Shard 90/128
Shard 91/128
Shard 92/128
Shard 93/128
Shard 94/128
Shard 95/128
Shard 96/128
Shard 97/128
Shard 98/128
Shard 99/128
Shard 100/128
Shard 101/128
Shard 102/128
Shard 103/128
Shard 104/128
Shard 105/128
Shard 106/128
Shard 107/128
Shard 108/128
Shard 109/128
Shard 110/128
Shard 111/128
Shard 112/128
Shard 113/128
Shard 114/128
Shard 115/128
Shard 116/128
Shard 117/128
Shard 118/128
Shard 119/128
Shard 120/128
Shard 121/128
Shard 122/128
Shard 123/128
Shard 124/128
Shard 125/128
Shard 126/128
Shard 127/128
Shard 128/128


In [19]:
user_items_csr = build_final_csr(
    out_dir=OUT_DIR,
    n_shards=N_SHARDS,
)

user_items_csr

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 272546528 stored elements and shape (951404, 4247685)>

### Обучение ALS

In [20]:
user_items_csr = load_npz(f"{OUT_DIR}/interaction_matrix.npz")
user_items_csr

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 272546528 stored elements and shape (951404, 4247685)>

In [22]:
model = AlternatingLeastSquares(
    factors=128,
    regularization=0.01,
    iterations=20,
    use_gpu=False,
)

model.fit(user_items_csr)

  0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
model.save("../models/user_mapping.pkl")

user_map = pd.read_parquet(f"{OUT_DIR}/user_mapping.parquet")
item_map = pd.read_parquet(f"{OUT_DIR}/item_mapping.parquet")

user_map.to_parquet(f"../models/als_user_mapping.parquet", index=False)
item_map.to_parquet(f"../models/als_item_mapping.parquet", index=False)

### Подготовка ground truth

In [64]:
orders_ddf_test = load_orders(train=False)
eval_df, gt, users_for_eval = make_df_for_eval(
    orders_ddf_test, OUT_DIR, NEW_VAL_START, NEW_VAL_END)

display(eval_df.head())
print("Users in val GT:", len(gt))
print("Avg GT size:", np.mean([len(v) for v in gt.values()]))

,user_id,item_id,u,i
0,981,7510328,184,93626
1,4670,136809690,887,1713853
2,12341,232878073,2323,2915706
3,12860,42322137,2422,530330
4,15330,22480781,2887,281708


Users in val GT: 345318
Avg GT size: 4.12001401606635


In [65]:
user_map = pd.read_parquet(f"{OUT_DIR}/user_mapping.parquet")
item_map = pd.read_parquet(f"{OUT_DIR}/item_mapping.parquet")

purchased_by_user = build_purchased_by_user_idx(
    orders_ddf, user_map, item_map, NEW_TRAIN_START, NEW_TRAIN_END)

purchased_by_user

{np.int32(439877): array([   1457, 3243652, 2954297, 3766754,  208036,   80608], dtype=int32),
 np.int32(652383): array([   1457, 1498179, 1053211,  561259, 1976529, 2673634,  785800,
         205022, 3555204, 1850038, 1500186, 2251999, 4114528, 3159635,
        2205905,  388724, 1971881, 1181079,   41604, 1337295, 1992543,
        2121061], dtype=int32),
 np.int32(545580): array([   2379, 2125469, 4195138, 3020896, 3087449, 3414980, 1203624,
        3496404, 3064390,  418161, 1484589, 1178239, 2095747, 2040762,
        3086021, 3486830, 2596811, 1371866, 2016412, 2707292, 3661787,
        1835876,  795708, 2025405, 3559248, 2650500, 3763467, 3814855,
        1243483,  987719, 4091123, 4208315, 1865549, 1714531, 1724995,
        3093161, 2320376, 1979238,  709746,  325713,   95468, 2965682,
        3602894, 3786046,  473042, 2858889, 1016387, 1244984, 3384925,
        1187068, 2790964, 3458456], dtype=int32),
 np.int32(209013): array([   2379, 1054445,  647713, 4005055, 2813179, 217637

In [ ]:
normalized_purchased_by_user_idx = {
    int(user_idx): items.astype(np.int32, copy=False)
    for user_idx, items in purchased_by_user.items()
}

np.save(
    f"../models/als_purchased_by_user_idx.npy"
    normalized_purchased_by_user_idx,
    allow_pickle=True
)

### Оценка на тестовой выборке

In [66]:
NDCG_100 = evaluate_ndcg(
    model=model,
    matrix=user_items_csr,
    users_for_eval=users_for_eval,
    gt=gt,
    purchased_by_user=purchased_by_user,
    k=100,
    batch_size=512,
    oversample=1000,
)

print(f"NDCG@100: {NDCG_100:.5f}")

100%|██████████| 675/675 [25:42<00:00,  2.29s/it]

NDCG@100: 0.01183
